<a href="https://colab.research.google.com/github/sanikaa12-art/hinglish-sarcasm-muril/blob/main/hinglish_sarcasm_muril.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone -q https://github.com/sahilswami96/SarcasmDetection_CodeMixed
import pandas as pd
from sklearn.model_selection import train_test_split
D = 'SarcasmDetection_CodeMixed/Dataset/'

def blocks(path):
    raw = open(path, encoding='utf-8').read().strip().split('\n\n')
    return [b.strip().split('\n') for b in raw if b.strip()]
texts = {b[0].strip(): ' '.join(b[1:]).strip() for b in blocks(D + 'Sarcasm_tweets.txt')}
lines = [l.strip() for l in open(D + 'Sarcasm_tweet_truth.txt', encoding='utf-8') if l.strip()]
labels = {lines[i]: lines[i+1] for i in range(0, len(lines), 2)}

df = pd.DataFrame([(k, texts[k], 1 if labels[k] == 'YES' else 0) for k in texts], columns=['id','text','label'])
train, temp = train_test_split(df, test_size=0.30, stratify=df.label, random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp.label, random_state=42)
print(len(train), len(val), len(test)); print(train.label.mean(), test.label.mean())
df.sample(5, random_state=1)

3675 787 788
0.09605442176870749 0.09644670050761421


,id,text,label
4,866264330748219392,#RememberingRajiv aaj agar musalman auraten tr...,0
3248,906106087945928704,Naamwar Bollywood adakaara Sunny Leone ka kehn...,0
440,895662711488434176,"From ""Gadar - Ek prem katha"" to ""Toilet - ek p...",0
1413,869608307434115072,"Abe suar ki aulad,TRIPLE TALAQ nd HALALA ki sa...",0
4351,899994002161795072,Yaar kabhi toh minority appeasment se upar uth...,0


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=2)
Xtr, Xte = vec.fit_transform(train.text), vec.transform(test.text)
clf = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr, train.label)
pred = clf.predict(Xte)

print('Baseline macro-F1:', round(f1_score(test.label, pred, average='macro'), 4))
print(classification_report(test.label, pred, digits=3, target_names=['not sarcastic','sarcastic']))

Baseline macro-F1: 0.8954
               precision    recall  f1-score   support

not sarcastic      0.999     0.954     0.976       712
    sarcastic      0.694     0.987     0.815        76

     accuracy                          0.957       788
    macro avg      0.846     0.970     0.895       788
 weighted avg      0.969     0.957     0.960       788



In [3]:
import re
LEAK = re.compile(r'#?\b(sarcas\w*|irony|ironic\w*)\b', re.I)
for d in (train, val, test):
    d['clean'] = d.text.apply(lambda t: LEAK.sub(' ', t).strip())

print('Leak words left:', sum(test.clean.str.contains(LEAK)))

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=2)
Xtr, Xte = vec.fit_transform(train.clean), vec.transform(test.clean)
clf = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr, train.label)
pred = clf.predict(Xte)
print('Clean baseline macro-F1:', round(f1_score(test.label, pred, average='macro'), 4))
print(classification_report(test.label, pred, digits=3, target_names=['not sarcastic','sarcastic']))

/tmp/ipykernel_485/4019678907.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print('Leak words left:', sum(test.clean.str.contains(LEAK)))


Leak words left: 0
Clean baseline macro-F1: 0.8882
               precision    recall  f1-score   support

not sarcastic      0.996     0.954     0.974       712
    sarcastic      0.689     0.961     0.802        76

     accuracy                          0.954       788
    macro avg      0.842     0.957     0.888       788
 weighted avg      0.966     0.954     0.958       788



In [4]:
for d in (train, val, test):
    d['clean'] = d.clean.str.replace(r'\s+', ' ', regex=True).str.strip()


In [6]:
!pip -q install transformers datasets accelerate
import torch, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report

MODEL = 'google/muril-base-cased'
tok = AutoTokenizer.from_pretrained(MODEL)
enc = lambda b: tok(b['clean'], truncation=True, max_length=64, padding='max_length')
ds = {n: Dataset.from_pandas(d[['clean','label']].reset_index(drop=True)).map(enc, batched=True)
      for n, d in [('train',train),('val',val),('test',test)]}

w = torch.tensor([1.0, (train.label==0).sum()/(train.label==1).sum()], dtype=torch.float)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels'); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, pred, average='macro'), 'sarcasm_f1': f1_score(p.label_ids, pred)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments('muril-out', eval_strategy='epoch', save_strategy='epoch', learning_rate=2e-5,
    per_device_train_batch_size=32, num_train_epochs=4, load_best_model_at_end=True,
    metric_for_best_model='macro_f1', seed=42, report_to='none')
trainer = WeightedTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['val'], compute_metrics=metrics)
trainer.train()

p = trainer.predict(ds['test'])
print('MuRIL macro-F1:', round(p.metrics['test_macro_f1'], 4))
print(classification_report(test.label, p.predictions.argmax(-1), digits=3, target_names=['not sarcastic','sarcastic']))

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Map:   0%|          | 0/3675 [00:00<?, ? examples/s]

Map:   0%|          | 0/787 [00:00<?, ? examples/s]

... (9 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MuRIL macro-F1: 0.8875
               precision    recall  f1-score   support

not sarcastic      0.991     0.959     0.975       712
    sarcastic      0.707     0.921     0.800        76

     accuracy                          0.956       788
    macro avg      0.849     0.940     0.888       788
 weighted avg      0.964     0.956     0.958       788



In [7]:
!git clone -q https://github.com/LCS2-IIITD/MSH-COMICS
import pickle, numpy.core.multiarray as ncm
# Safe loader: only allows pandas/numpy objects (pickles can otherwise run code)
OK = {('builtins','slice'),('numpy','dtype'),('numpy','ndarray'),('numpy.core.multiarray','_reconstruct'),
      ('pandas.core.frame','DataFrame'),('pandas.core.indexes.base','Index'),('pandas.core.indexes.base','_new_Index'),
      ('pandas.core.indexes.range','RangeIndex'),('pandas.core.internals.managers','BlockManager')}
class SafeLoader(pickle.Unpickler):
    def find_class(self, m, n):
        if (m, n) not in OK: raise pickle.UnpicklingError(f'blocked {m}.{n}')
        return getattr(ncm, n) if m == 'numpy.core.multiarray' else super().find_class(m, n)

masac = SafeLoader(open('MSH-COMICS/Data/Pickles/Dataset.p','rb')).load()
masac = masac[['text','Sarcasm']].rename(columns={'Sarcasm':'label'}).dropna()
masac['clean'] = masac.text.str.replace(r'\s+', ' ', regex=True).str.strip()
ood = masac.sample(2000, random_state=42).reset_index(drop=True)
print(len(ood), ood.label.mean())

# Baseline trained on tweets -> tested on TV dialogue
bp = clf.predict(vec.transform(ood.clean))
print('Baseline cross-domain macro-F1:', round(f1_score(ood.label, bp, average='macro'), 4))

# MuRIL trained on tweets -> tested on TV dialogue
ood_ds = Dataset.from_pandas(ood[['clean','label']]).map(enc, batched=True)
mp = trainer.predict(ood_ds).predictions.argmax(-1)
print('MuRIL cross-domain macro-F1:', round(f1_score(ood.label, mp, average='macro'), 4))
print(classification_report(ood.label, mp, digits=3, target_names=['not sarcastic','sarcastic']))

2000 0.205
Baseline cross-domain macro-F1: 0.215


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

MuRIL cross-domain macro-F1: 0.1825
               precision    recall  f1-score   support

not sarcastic      0.697     0.014     0.028      1590
    sarcastic      0.203     0.976     0.337       410

     accuracy                          0.211      2000
    macro avg      0.450     0.495     0.182      2000
 weighted avg      0.596     0.211     0.092      2000

